# 🍜 STEP 03 — 멀티 에이전트: AgentGroupChat + 트리아지 오케스트레이터

**STEP 02**에서 만든 3개 전문 에이전트를 하나의 그룹으로 묶고,
트리아지 에이전트(오케스트레이터)가 요청을 적절한 에이전트에 라우팅합니다.

---

## 이 노트북에서 배울 것

| 개념 | 설명 |
|------|------|
| `AgentGroupChat` | 여러 에이전트가 참여하는 대화 그룹 |
| `KernelFunctionSelectionStrategy` | 다음 에이전트를 LLM 프롬프트로 선택 |
| `KernelFunctionTerminationStrategy` | 종료 조건을 LLM 프롬프트로 판단 |
| Plugin-as-Agent 패턴 | 에이전트를 다른 에이전트의 플러그인으로 등록 |

---

## F&B 시스템 흐름
```
사용자 입력
    ↓
TriageAgent (오케스트레이터) ← SelectionStrategy가 제어
    ├→ LegalTaxAgent  (법령·세무 질문)
    ├→ LocationAgent  (상권 분석 질문)
    └→ FinanceAgent   (재무 분석 질문)
    ↓
TerminationStrategy가 "완료" 판단 → 최종 응답
```

In [ ]:
import asyncio, os, json
from typing import Annotated
from dotenv import load_dotenv

from semantic_kernel import Kernel
from semantic_kernel.agents import AgentGroupChat, ChatCompletionAgent
from semantic_kernel.agents.strategies import (
    KernelFunctionSelectionStrategy,
    KernelFunctionTerminationStrategy,
)
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion, AzureChatPromptExecutionSettings
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.contents import ChatHistoryTruncationReducer
from semantic_kernel.functions import kernel_function, KernelArguments, KernelFunctionFromPrompt

load_dotenv()
print("임포트 완료")

In [ ]:
# STEP 02와 동일한 헬퍼 + 플러그인 재사용
def make_kernel() -> Kernel:
    kernel = Kernel()
    kernel.add_service(AzureChatCompletion(
        deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
        endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2025-03-01-preview"),
    ))
    return kernel

def make_auto_settings():
    s = AzureChatPromptExecutionSettings()
    s.function_choice_behavior = FunctionChoiceBehavior.Auto()
    return s

class FnbLegalPlugin:
    @kernel_function(description="F&B 업종별 영업 인허가 요건을 반환합니다.")
    def get_license_requirements(self, business_type: Annotated[str, "업종"], area_sqm: Annotated[float, "면적(㎡)"]) -> str:
        docs = {"일반음식점": "영업신고(보건소)·위생교육(6h)·소방완비증명(300㎡↑)",
                "카페": "영업신고(보건소)·위생교육(3h)"}
        return docs.get(business_type, "해당 정보 없음")

    @kernel_function(description="연매출에 따른 과세유형을 안내합니다.")
    def get_tax_type_guide(self, annual_revenue_krw: Annotated[int, "연 매출액(원)"]) -> str:
        return "간이과세" if annual_revenue_krw < 104_000_000 else "일반과세"

class FnbLocationPlugin:
    @kernel_function(description="위치와 업종에 따른 상권 분석을 반환합니다.")
    def analyze_commercial_area(self, location: Annotated[str, "위치"], business_type: Annotated[str, "업종"]) -> str:
        return json.dumps({"location": location, "daily_population": 45000, "competitors_500m": 12, "rent_per_sqm": 85000}, ensure_ascii=False)

class FnbFinancePlugin:
    @kernel_function(description="초기투자비, 고정비, 객단가, 변동비율로 BEP를 계산합니다.")
    def calculate_bep(self,
        initial_investment_krw: Annotated[int, "초기투자비(원)"],
        monthly_fixed_cost_krw: Annotated[int, "월고정비(원)"],
        avg_revenue_per_customer_krw: Annotated[int, "객단가(원)"],
        variable_cost_ratio: Annotated[float, "변동비율(0~1)"]
    ) -> str:
        cm = 1 - variable_cost_ratio
        bep_revenue = monthly_fixed_cost_krw / cm
        bep_customers = bep_revenue / avg_revenue_per_customer_krw
        return "월 BEP 매출: %s원 / 월 BEP 고객: %d명" % ("{:,.0f}".format(bep_revenue), int(bep_customers))

print("헬퍼·플러그인 정의 완료")

---
## 1. 에이전트 3종 생성 (STEP 02 복습)

In [ ]:
# ── LegalTaxAgent ──
legal_k = make_kernel()
legal_k.add_plugin(FnbLegalPlugin(), plugin_name="FnbLegal")
legal_tax_agent = ChatCompletionAgent(
    name="LegalTaxAgent",
    description="F&B 인허가·법령·세무 전문 에이전트",
    instructions="F&B 창업 법령·세무 전문가. 관련 법조항을 명시하며 정확한 정보 제공.",
    kernel=legal_k,
    arguments=KernelArguments(settings=make_auto_settings()),
)

# ── LocationAgent ──
loc_k = make_kernel()
loc_k.add_plugin(FnbLocationPlugin(), plugin_name="FnbLocation")
location_agent = ChatCompletionAgent(
    name="LocationAgent",
    description="유동인구·경쟁업체·임대료 등 상권 분석 전문 에이전트",
    instructions="F&B 상권 분석 전문가. 수치 데이터와 함께 리스크/기회 요소를 분석.",
    kernel=loc_k,
    arguments=KernelArguments(settings=make_auto_settings()),
)

# ── FinanceAgent ──
fin_k = make_kernel()
fin_k.add_plugin(FnbFinancePlugin(), plugin_name="FnbFinance")
finance_agent = ChatCompletionAgent(
    name="FinanceAgent",
    description="BEP·시나리오 분석·리스크 시뮬레이션 전문 에이전트",
    instructions="F&B 재무 분석 전문가. 낙관/기본/비관 3시나리오를 항상 포함.",
    kernel=fin_k,
    arguments=KernelArguments(settings=make_auto_settings()),
)

print("에이전트 3종 생성 완료")

---
## 2. SelectionStrategy — 다음 에이전트를 LLM이 선택

> **핵심 개념**: 누가 다음에 말할지를 LLM 프롬프트로 결정합니다.
> 프롬프트가 에이전트 이름만 반환하면 SK가 해당 에이전트를 호출합니다.

In [ ]:
AGENT_NAMES = ["LegalTaxAgent", "LocationAgent", "FinanceAgent"]

# SelectionStrategy 프롬프트
# {{$agents}}   : 참여 에이전트 목록 (SK가 자동 주입)
# {{$history}}  : 대화 기록 (SK가 자동 주입)
selection_prompt = """
당신은 F&B 창업 지원 멀티 에이전트 시스템의 라우터입니다.
아래 대화 기록을 분석하여 다음에 응답할 에이전트 이름 하나만 반환하세요.

[에이전트 목록]
{{$agents}}

[라우팅 규칙]
- 인허가, 위생교육, 영업신고, 세무, 과세유형 → LegalTaxAgent
- 상권, 유동인구, 경쟁업체, 임대료, 입지, 위치 → LocationAgent  
- 투자비, BEP, 손익분기점, 매출 예측, 리스크 → FinanceAgent
- 복합 요청이면 먼저 처리해야 할 것부터 선택

[대화 기록]
{{$history}}

위 규칙에 따라 다음에 응답할 에이전트 이름만 반환하세요 (설명 없이):
"""

# TerminationStrategy 프롬프트  
# "완료" 또는 지정된 키워드가 포함되면 대화 종료
termination_prompt = """
아래 대화에서 사용자의 질문이 충분히 답변되었는지 판단하세요.

[판단 기준]
- 전문 에이전트가 구체적인 답변을 제공했다 → 종료
- 아직 답변이 불완전하거나 후속 작업이 필요하다 → 계속

[대화 기록]
{{$history}}

종료 여부를 판단하여 "완료" 또는 "계속" 중 하나만 반환하세요:
"""

# Kernel (전략 판단용 — 플러그인 없이 순수 LLM 판단)
strategy_kernel = make_kernel()

selection_function = KernelFunctionFromPrompt(
    function_name="agent_selection",
    prompt=selection_prompt,
)

termination_function = KernelFunctionFromPrompt(
    function_name="termination_check",
    prompt=termination_prompt,
)

print("전략 프롬프트 정의 완료")

---
## 3. AgentGroupChat 구성 — 오케스트레이터 완성

In [ ]:
def create_fnb_group_chat() -> AgentGroupChat:
    """
    F&B 창업 지원 AgentGroupChat을 생성합니다.

    구성:
    - 참여 에이전트: LegalTaxAgent, LocationAgent, FinanceAgent
    - SelectionStrategy: LLM이 다음 에이전트를 선택
    - TerminationStrategy: LLM이 종료 여부를 판단
    - 최대 반복: 10회 (무한 루프 방지)
    """
    # 대화 기록을 너무 길게 보내지 않도록 마지막 5개 메시지만 전략 프롬프트에 포함
    history_reducer = ChatHistoryTruncationReducer(target_count=5)

    selection_strategy = KernelFunctionSelectionStrategy(
        function=selection_function,
        kernel=strategy_kernel,
        # 프롬프트 결과에서 에이전트 이름을 파싱하는 함수
        result_parser=lambda result: str(result.value[0]).strip(),
        history_variable_name="history",
        history_reducer=history_reducer,
    )

    termination_strategy = KernelFunctionTerminationStrategy(
        function=termination_function,
        kernel=strategy_kernel,
        # "완료" 텍스트가 응답에 포함되면 대화 종료
        result_parser=lambda result: "완료" in str(result.value[0]),
        history_variable_name="history",
        history_reducer=history_reducer,
        maximum_iterations=10,  # 안전장치: 최대 10회 반복
    )

    return AgentGroupChat(
        agents=[legal_tax_agent, location_agent, finance_agent],
        selection_strategy=selection_strategy,
        termination_strategy=termination_strategy,
    )

print("AgentGroupChat 팩토리 함수 정의 완료")

---
## 4. 라우팅 테스트 — 각 전문 에이전트로 자동 라우팅

In [ ]:
from semantic_kernel.contents import ChatMessageContent, AuthorRole

async def ask_group(question: str):
    """GroupChat에 질문을 던지고 에이전트 응답을 출력합니다."""
    chat = create_fnb_group_chat()  # 매번 새 GroupChat 생성 (세션 초기화)

    # 사용자 메시지 추가
    await chat.add_chat_message(
        ChatMessageContent(role=AuthorRole.USER, content=question)
    )

    print("[질문] %s" % question)
    print("=" * 60)

    # 에이전트 응답 스트리밍
    async for msg in chat.invoke():
        print("[%s]" % msg.name)
        print(msg.content)
        print("-" * 60)


# 테스트 1: 법령 질문 → LegalTaxAgent로 라우팅되어야 함
await ask_group("서울에서 30평짜리 일반음식점을 열려고 합니다. 위생 인허가 절차를 알려주세요.")

In [ ]:
# 테스트 2: 상권 질문 → LocationAgent로 라우팅되어야 함
await ask_group("홍대입구역 근처 카페 상권이 어떤지 분석해줘. 유동인구와 경쟁업체 현황이 궁금해.")

In [ ]:
# 테스트 3: 재무 질문 → FinanceAgent로 라우팅되어야 함
await ask_group("초기 투자비 8천만원, 월 고정비 350만원, 객단가 7000원, 재료비율 35%일 때 손익분기점 알려줘.")

In [ ]:
# 테스트 4: 복합 질문 → 순차 라우팅 (LegalTaxAgent → FinanceAgent)
await ask_group(
    "계약서 검토 후 영업신고서 작성을 도와줘. "
    "그리고 월 임대료 300만원을 고정비로 잡았을 때 BEP도 같이 계산해줘."
)

---
## 5. [심화] Plugin-as-Agent 패턴

> **다른 접근법**: 에이전트를 `GroupChat` 대신 트리아지 에이전트의 **플러그인**으로 등록하는 방법.
> 
> - 장점: 단순함, 트리아지 에이전트가 직접 결과를 합성 가능
> - 단점: 병렬 호출 어려움, 각 에이전트 결과를 별도 스트리밍 불가
> 
> 아키텍처 다이어그램의 `HandoffBuilder` 패턴과 유사합니다.

In [ ]:
# Plugin-as-Agent: 에이전트를 플러그인으로 래핑
class AgentAsPlugin:
    """에이전트를 플러그인 함수로 래핑하는 어댑터."""

    def __init__(self, agent: ChatCompletionAgent):
        self._agent = agent

    @kernel_function(description="법령·세무 전문 에이전트에게 질문을 위임합니다.")
    async def ask_legal_tax(
        self,
        question: Annotated[str, "법령·세무 관련 질문"],
    ) -> str:
        response = await self._agent.get_response(messages=question)
        return response.content


# 트리아지 에이전트: 하위 에이전트 플러그인을 직접 호출
triage_kernel = make_kernel()
triage_kernel.add_plugin(
    AgentAsPlugin(legal_tax_agent),
    plugin_name="SubAgents",
)

triage_agent = ChatCompletionAgent(
    name="TriageAgent",
    description="사용자 요청을 분류하고 전문 에이전트로 위임하는 오케스트레이터",
    instructions=(
        "당신은 F&B 창업 지원 어시스턴트입니다. "
        "법령·세무 질문은 ask_legal_tax 함수를 호출하여 위임하세요."
    ),
    kernel=triage_kernel,
    arguments=KernelArguments(settings=make_auto_settings()),
)

response = await triage_agent.get_response(
    messages="카페 창업할 때 위생교육은 몇 시간 들어야 해?"
)
print("[TriageAgent (Plugin-as-Agent 패턴) 응답]")
print(response.content)

---
## 6. ✅ STEP 03 정리

| 배운 것 | 코드 패턴 | 다음 단계 연결 |
|---------|-----------|---------------|
| AgentGroupChat | `AgentGroupChat(agents=[...])` | STEP 04 MCP 툴 연동 |
| SelectionStrategy | `KernelFunctionSelectionStrategy` | 라우팅 로직의 핵심 |
| TerminationStrategy | `KernelFunctionTerminationStrategy` | 종료 조건 제어 |
| Plugin-as-Agent | 에이전트 → 플러그인 래핑 | HandoffBuilder 대안 패턴 |

---

### 다음 STEP 미리보기
```
STEP 04: MCP (Model Context Protocol) 연동
  - 카카오 지도 API를 MCP 서버로 연동
  - 국가법령정보센터 API를 MCP 툴로 등록
  - SK에서 MCP 서버 호출 패턴
```